# Amazon Reviews — EDA & Model Evaluation

Visual analysis of the recommender system pipeline: data distribution, sparsity, long-tail phenomenon, and model comparison.

**Run all cells:** `Run > Run All Cells`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from scipy.sparse import csr_matrix, load_npz
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
%matplotlib inline

print('Libraries loaded')

In [ ]:
# Load raw data (only essential columns)
df = pd.read_csv('Reviews.csv', usecols=['UserId', 'ProductId', 'Score'])
print(f'Total records:  {len(df):,}')
print(f'Unique users:   {df["UserId"].nunique():,}')
print(f'Unique products: {df["ProductId"].nunique():,}')
print(f'Sparsity:       {1 - len(df) / (df["UserId"].nunique() * df["ProductId"].nunique()):.6f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- Score histogram ---
colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#27ae60']
sns.countplot(data=df, x='Score', palette=colors, ax=axes[0], edgecolor='white')
axes[0].set_title('Score Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')
for p in axes[0].containers:
    axes[0].bar_label(p, fmt='%,d', fontsize=9)

# --- Pie chart ---
score_pct = df['Score'].value_counts().sort_index()
axes[1].pie(score_pct.values, labels=[f'{s}★\n{p:.1f}%' for s, p in zip(score_pct.index, score_pct.values/score_pct.sum()*100)],
            colors=colors, startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Score Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Mean score: {df["Score"].mean():.4f}  |  Median: {df["Score"].median():.1f}  |  5-star rate: {(df["Score"]==5).mean()*100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

user_counts = df['UserId'].value_counts()
item_counts = df['ProductId'].value_counts()

# --- User long tail ---
axes[0].plot(range(1, len(user_counts)+1), sorted(user_counts.values, reverse=True),
             color='#3498db', linewidth=0.8, alpha=0.8)
axes[0].axhline(y=5, color='red', linestyle='--', alpha=0.6, label='Threshold (5)')
axes[0].set_title('User Activity — Long Tail', fontsize=14, fontweight='bold')
axes[0].set_xlabel('User Rank (sorted by # reviews)')
axes[0].set_ylabel('Number of Reviews')
axes[0].set_yscale('log')
axes[0].set_xscale('log')
axes[0].legend()
axes[0].annotate(f'{(user_counts==1).mean()*100:.1f}% users have only 1 review',
                xy=(0.02, 0.85), xycoords='axes fraction', fontsize=10,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.3))

# --- Item long tail ---
axes[1].plot(range(1, len(item_counts)+1), sorted(item_counts.values, reverse=True),
             color='#e74c3c', linewidth=0.8, alpha=0.8)
axes[1].axhline(y=5, color='red', linestyle='--', alpha=0.6, label='Threshold (5)')
axes[1].set_title('Product Popularity — Long Tail', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Product Rank (sorted by # reviews)')
axes[1].set_ylabel('Number of Reviews')
axes[1].set_yscale('log')
axes[1].set_xscale('log')
axes[1].legend()
axes[1].annotate(f'{(item_counts==1).mean()*100:.1f}% products have only 1 review',
                xy=(0.02, 0.85), xycoords='axes fraction', fontsize=10,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.3))

plt.suptitle('Long Tail Analysis (log-log scale)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
df_f = df.copy()
valid_users = df_f['UserId'].value_counts()[lambda x: x >= 5].index
df_f = df_f[df_f['UserId'].isin(valid_users)]
valid_items = df_f['ProductId'].value_counts()[lambda x: x >= 5].index
df_f = df_f[df_f['ProductId'].isin(valid_items)]
df_f = df_f.drop_duplicates(subset=['UserId', 'ProductId'], keep='last')

metrics = pd.DataFrame({
    'Metric': ['Records', 'Users', 'Products', 'Sparsity'],
    'Before': [f'{len(df):,}', f'{df["UserId"].nunique():,}', f'{df["ProductId"].nunique():,}',
               f'{1 - len(df)/(df["UserId"].nunique()*df["ProductId"].nunique()):.4%}'],
    'After': [f'{len(df_f):,}', f'{df_f["UserId"].nunique():,}', f'{df_f["ProductId"].nunique():,}',
              f'{1 - len(df_f)/(df_f["UserId"].nunique()*df_f["ProductId"].nunique()):.4%}']
})

fig, ax = plt.subplots(figsize=(8, 3))
ax.axis('off')
tbl = ax.table(cellText=metrics.values, colLabels=metrics.columns,
               loc='center', cellLoc='center',
               colColours=['#2c3e50', '#3498db', '#27ae60'])
tbl.auto_set_font_size(False)
tbl.set_fontsize(12)
tbl.scale(1, 2)
ax.set_title('Filtering Impact', fontsize=14, fontweight='bold', pad=20)
plt.show()

print(f'Data reduction: {len(df_f)/len(df):.1%} records retained')
print(f'Density improvement: {(1 - len(df_f)/(df_f["UserId"].nunique()*df_f["ProductId"].nunique())) / (1 - len(df)/(df["UserId"].nunique()*df["ProductId"].nunique()):.4f}x')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

user_counts_f = df_f['UserId'].value_counts()
item_counts_f = df_f['ProductId'].value_counts()

axes[0].hist(user_counts_f, bins=50, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].set_title(f'User Review Counts (after filter)\nMedian={user_counts_f.median():.0f}, Max={user_counts_f.max():,}',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Reviews per User')
axes[0].set_ylabel('User Count')

axes[1].hist(item_counts_f, bins=50, color='#e74c3c', edgecolor='white', alpha=0.8)
axes[1].set_title(f'Product Review Counts (after filter)\nMedian={item_counts_f.median():.0f}, Max={item_counts_f.max():,}',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Reviews per Product')
axes[1].set_ylabel('Product Count')

plt.tight_layout()
plt.show()

In [ ]:
# Model RMSE from the recommender pipeline
model_results = pd.DataFrame({
    'Model': ['Baseline', 'Item-Based CF', 'User-Based CF', 'TruncatedSVD'],
    'RMSE': [0.9341, 0.7369, 1.3880, 0.9802],
    'Type': ['Bias-only', 'Neighborhood', 'Neighborhood', 'Latent Factor']
})

fig, ax = plt.subplots(figsize=(10, 5))

bar_colors = ['#95a5a6', '#27ae60', '#e74c3c', '#f39c12']
bars = ax.bar(model_results['Model'], model_results['RMSE'], color=bar_colors,
              edgecolor='white', linewidth=1.5, width=0.6)

# Value labels
for bar, val in zip(bars, model_results['RMSE']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.4f}', ha='center', va='bottom', fontsize=13, fontweight='bold')

# Baseline reference line
ax.axhline(y=0.9341, color='gray', linestyle='--', alpha=0.5, linewidth=1)
ax.text(3.5, 0.9341, 'Baseline', ha='right', va='bottom', fontsize=9, color='gray')

# Improvement annotations
ax.annotate('-21.1% 🎯', xy=(1, 0.7369), xytext=(1, 0.65),
            fontsize=12, fontweight='bold', color='#27ae60',
            ha='center',
            arrowprops=dict(arrowstyle='->', color='#27ae60', lw=2))

ax.set_ylabel('RMSE (lower is better)', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=15, fontweight='bold')
ax.set_ylim(0, 1.6)
# Add type labels under model names
for i, t in enumerate(model_results['Type']):
    ax.text(i, -0.12, t, ha='center', fontsize=9, color='#555', style='italic')

plt.tight_layout()
plt.show()

print('Best model: Item-Based CF (RMSE = 0.7369)')
print(f'Improvement over baseline: {(0.9341-0.7369)/0.9341*100:.1f}%')

In [ ]:
import pickle
import os

svd_path = 'models/svd.pkl'
if os.path.exists(svd_path):
    with open(svd_path, 'rb') as f:
        svd = pickle.load(f)
    test_indices = np.load('data/test_indices.npy')
    test_df = pd.DataFrame(test_indices, columns=['u_id', 'i_id', 'Score']).astype(int)
    preds = svd.predict(test_df)
    errors = preds - test_df['Score'].values

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Histogram of errors
    axes[0].hist(errors, bins=50, color='#3498db', edgecolor='white', alpha=0.8)
    axes[0].axvline(x=0, color='red', linestyle='--', alpha=0.6)
    axes[0].set_title(f'SVD Prediction Error Distribution\nRMSE = {np.sqrt(np.mean(errors**2)):.4f}',
                     fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Prediction Error (predicted - actual)')
    axes[0].set_ylabel('Count')

    # Actual vs Predicted scatter
    sample_idx = np.random.choice(len(test_df), min(5000, len(test_df)), replace=False)
    axes[1].scatter(test_df['Score'].values[sample_idx], preds[sample_idx],
                   alpha=0.3, s=10, color='#2ecc71', edgecolor='white', linewidth=0.3)
    axes[1].plot([1, 5], [1, 5], 'r--', alpha=0.5, linewidth=1)
    axes[1].set_title('Actual vs Predicted (SVD)', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Actual Rating')
    axes[1].set_ylabel('Predicted Rating')
    axes[1].set_xlim(0.5, 5.5)
    axes[1].set_ylim(0.5, 5.5)

    plt.tight_layout()
    plt.show()
else:
    print('Model files not found. Run recommender.py first to generate them.')
    print('\nYou can still run the EDA cells above without model outputs.')

## Key Insights

| Finding | Detail |
|---------|--------|
| **Severe class imbalance** | 63.9% of ratings are 5 stars — model has positive bias |
| **Extreme long tail** | 68.5% of users & 40.9% of products have only 1 review |
| **Sparsity** | 99.997% — only 3 out of every 100,000 matrix cells are filled |
| **ItemCF best** | RMSE 0.7369, 21.1% better than baseline |
| **UserCF unusable** | RMSE 1.388 — worse than predicting the global mean |
| **SVD limited** | Only 27.8% residual variance explained; rating variance too low |

### Recommendations for improvement
- **Debiasing**: Center ratings per user to reduce the 5-star bias
- **Hybrid**: Combine ItemCF + SVD (SVD++ style) for complementary strengths
- **Implicit feedback**: Incorporate helpfulness votes as auxiliary signal
- **Ranking loss**: Switch from RMSE to NDCG/LambdaRank for list quality
- **ANN search**: Replace O(n²) similarity matrix with Faiss for scalability